In [1]:
!pip install -q langgraph langchain-groq langchain-core groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [2]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Annotated
import operator
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("✅ LangGraph setup complete")

✅ LangGraph setup complete


In [3]:
# State is what gets passed between nodes
# Think of it as the agent's memory

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    # Annotated[list, operator.add] means:
    # - messages is a list
    # - when updating, ADD new messages to existing ones
    # - not replace them
    # This gives the agent conversation memory!

print("✅ State defined")
print("""
AgentState explanation:

Every node in our graph receives the current state
and returns an updated state.

messages = full conversation history
Each step ADDS to history, never replaces it.

This is why LangGraph has memory —
the state carries everything forward.
""")

✅ State defined

AgentState explanation:

Every node in our graph receives the current state
and returns an updated state.

messages = full conversation history
Each step ADDS to history, never replaces it.

This is why LangGraph has memory —
the state carries everything forward.



In [4]:
@tool
def check_customer_risk(balance: float, transactions: int, is_active: bool) -> str:
    """Check the risk level of a banking customer.
    Use when asked about customer risk assessment."""
    if not is_active:
        return "HIGH RISK — Account inactive"
    if balance < 1000:
        return f"HIGH RISK — Low balance: €{balance}"
    elif balance < 5000:
        return f"MEDIUM RISK — Moderate balance: €{balance}"
    else:
        return f"LOW RISK — Healthy balance: €{balance}"

@tool
def check_aml_flag(transaction_amount: float, country: str) -> str:
    """Check if a transaction requires AML screening.
    Use when asked about AML or suspicious transactions."""
    high_risk_countries = ["Country_A", "Country_B", "Country_C"]
    flags = []
    if transaction_amount > 10000:
        flags.append("Transaction above EUR 10,000 — monitoring required")
    if country in high_risk_countries:
        flags.append(f"{country} is high risk — Enhanced Due Diligence required")
    if flags:
        return "🚨 AML FLAGS:\n" + "\n".join(flags)
    return "✅ No AML flags"

@tool
def check_credit_limit(client_exposure: float, total_portfolio: float, sector: str) -> str:
    """Check if client exposure breaches credit limits.
    Use when asked about credit limits or portfolio concentration."""
    exposure_pct = (client_exposure / total_portfolio) * 100
    issues = []
    if exposure_pct > 10:
        issues.append(f"⚠️ Exposure {exposure_pct:.1f}% exceeds 10% limit")
    if client_exposure > 5000000:
        issues.append("⚠️ Above EUR 5M — immediate escalation required")
    if issues:
        return "LIMIT BREACH:\n" + "\n".join(issues)
    return f"✅ Exposure {exposure_pct:.1f}% — within limits"

tools = [check_customer_risk, check_aml_flag, check_credit_limit]

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

print(f"✅ {len(tools)} tools created and bound to LLM")

✅ 3 tools created and bound to LLM


In [5]:
# Node 1 — The agent node
# This is where the LLM thinks and decides
def agent_node(state: AgentState):
    print("🤖 Agent thinking...")

    system = SystemMessage(content="""You are a senior banking compliance officer.
    Use tools to check customer risk, AML flags and credit limits.
    After getting tool results — give a clear final answer.
    Do NOT call the same tool twice.""")

    # Get all messages including history
    messages = [system] + state["messages"]

    # LLM decides — call a tool or give final answer
    response = llm_with_tools.invoke(messages)

    print(f"   Tool calls: {len(response.tool_calls)} requested")

    # Return updated state — adds response to messages
    return {"messages": [response]}


# Node 2 — The tool node
# This is where tools actually run
tool_node = ToolNode(tools)
# ToolNode is LangGraph's built in tool executor
# It automatically:
# - Reads which tool to call from agent response
# - Calls the tool with right arguments
# - Returns result as ToolMessage
# No manual for loop needed like Day 26!

print("✅ Nodes defined")
print("""
Two nodes in our graph:

agent_node — LLM thinks, decides what to do
tool_node  — executes the tool the LLM chose

These two nodes will pass messages back and forth
until the LLM decides no more tools are needed.
""")

✅ Nodes defined

Two nodes in our graph:

agent_node — LLM thinks, decides what to do
tool_node  — executes the tool the LLM chose

These two nodes will pass messages back and forth
until the LLM decides no more tools are needed.



In [6]:
# This function decides where to go next
def should_continue(state: AgentState):
    """
    After agent_node runs — where do we go?

    If agent called a tool → go to tool_node
    If agent gave final answer → go to END
    """
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        print("   → Going to tool_node")
        return "tools"
    else:
        print("   → Going to END")
        return END

print("✅ Routing logic defined")
print("""
should_continue decides the flow:

Agent responds with tool call?
    YES → route to tool_node
    NO  → route to END

This is what prevents infinite loops!
The agent can only go to END
when it stops calling tools.
""")

✅ Routing logic defined

should_continue decides the flow:

Agent responds with tool call?
    YES → route to tool_node
    NO  → route to END

This is what prevents infinite loops!
The agent can only go to END
when it stops calling tools.



In [7]:
# Create the graph
graph = StateGraph(AgentState)

# Add nodes
graph.add_node("agent", agent_node)
graph.add_node("tools", tool_node)

# Set entry point — where graph starts
graph.set_entry_point("agent")

# Add edges — connections between nodes
graph.add_conditional_edges(
    "agent",          # from agent node
    should_continue,  # run this function to decide
    {
        "tools": "tools",  # if returns "tools" → go to tool_node
        END: END           # if returns END → stop
    }
)

# After tools always go back to agent
graph.add_edge("tools", "agent")

# Compile the graph
app = graph.compile()

print("✅ Graph compiled!")
print("""
Graph structure:

START
  ↓
[agent] → should_continue → [tools]
  ↑                            ↓
  └────────────────────────────┘
  (loops until no more tool calls)
  ↓
END
""")

✅ Graph compiled!

Graph structure:

START
  ↓
[agent] → should_continue → [tools]
  ↑                            ↓
  └────────────────────────────┘
  (loops until no more tool calls)
  ↓
END



In [8]:
def run_langgraph_agent(question):
    print(f"\n❓ Question: {question}")
    print("-" * 55)

    # Initial state — just the user question
    initial_state = {
        "messages": [HumanMessage(content=question)]
    }

    # Run the graph
    final_state = app.invoke(initial_state)

    # Get the last message — that's the final answer
    final_answer = final_state["messages"][-1].content

    print(f"\n✅ FINAL ANSWER:\n{final_answer}")
    return final_answer

# Test 1
run_langgraph_agent("Check risk for customer with balance €750, 142 transactions, active account")


❓ Question: Check risk for customer with balance €750, 142 transactions, active account
-------------------------------------------------------
🤖 Agent thinking...
   Tool calls: 1 requested
   → Going to tool_node
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END

✅ FINAL ANSWER:
The customer has been identified as high risk due to a low balance of €750.


'The customer has been identified as high risk due to a low balance of €750.'

In [9]:
# Test 2 — AML
run_langgraph_agent("Is a €15,000 transaction from Country_A flagged for AML?")


❓ Question: Is a €15,000 transaction from Country_A flagged for AML?
-------------------------------------------------------
🤖 Agent thinking...
   Tool calls: 1 requested
   → Going to tool_node
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END

✅ FINAL ANSWER:
The €15,000 transaction from Country_A is flagged for AML screening due to the high-risk country and the transaction amount exceeding €10,000. Enhanced Due Diligence is required for this transaction.


'The €15,000 transaction from Country_A is flagged for AML screening due to the high-risk country and the transaction amount exceeding €10,000. Enhanced Due Diligence is required for this transaction.'

In [10]:
# Test 3 — Credit limit
run_langgraph_agent("Client exposure €8M out of €50M portfolio in real estate — any breaches?")


❓ Question: Client exposure €8M out of €50M portfolio in real estate — any breaches?
-------------------------------------------------------
🤖 Agent thinking...
   Tool calls: 1 requested
   → Going to tool_node
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END

✅ FINAL ANSWER:
The client exposure of €8M out of a €50M portfolio in real estate does breach the credit limits, as it exceeds the 10% limit and is above the €5M threshold, requiring immediate escalation.


'The client exposure of €8M out of a €50M portfolio in real estate does breach the credit limits, as it exceeds the 10% limit and is above the €5M threshold, requiring immediate escalation.'

In [11]:
# Test 4 — Multi tool
run_langgraph_agent("""
Complete compliance check:
- Balance €3,500, 89 transactions, active
- Transaction €12,000 from Country_B
- Exposure €6M out of €40M in manufacturing
""")


❓ Question: 
Complete compliance check:
- Balance €3,500, 89 transactions, active
- Transaction €12,000 from Country_B
- Exposure €6M out of €40M in manufacturing

-------------------------------------------------------
🤖 Agent thinking...
   Tool calls: 3 requested
   → Going to tool_node
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END

✅ FINAL ANSWER:
Based on the results, the customer is considered MEDIUM RISK due to a moderate balance of €3,500. The transaction of €12,000 from Country_B requires monitoring and Enhanced Due Diligence as it exceeds €10,000 and Country_B is high risk. Additionally, the client exposure of €6M in the manufacturing sector exceeds the 10% limit of the total portfolio of €40M, requiring immediate escalation. Therefore, the final answer is that the customer poses a medium risk and requires further monitoring and due diligence.


'Based on the results, the customer is considered MEDIUM RISK due to a moderate balance of €3,500. The transaction of €12,000 from Country_B requires monitoring and Enhanced Due Diligence as it exceeds €10,000 and Country_B is high risk. Additionally, the client exposure of €6M in the manufacturing sector exceeds the 10% limit of the total portfolio of €40M, requiring immediate escalation. Therefore, the final answer is that the customer poses a medium risk and requires further monitoring and due diligence.'

In [12]:
# Test conversation memory
from langchain_core.messages import AIMessage

def run_conversation(questions):
    print("🏦 Multi-turn Compliance Conversation")
    print("=" * 55)

    # Start with empty history
    state = {"messages": []}

    for question in questions:
        print(f"\n❓ {question}")
        print("-" * 40)

        # Add new question to existing history
        state["messages"].append(HumanMessage(content=question))

        # Run graph with full history
        state = app.invoke(state)

        # Get answer
        answer = state["messages"][-1].content
        print(f"💬 {answer}")

# Test multi-turn conversation
run_conversation([
    "Check risk for client with balance €750, active account, 50 transactions",
    "What was the risk level you just assessed?",
    "Now check AML for a €12,000 transaction from Country_A for that same client"
])

🏦 Multi-turn Compliance Conversation

❓ Check risk for client with balance €750, active account, 50 transactions
----------------------------------------
🤖 Agent thinking...
   Tool calls: 1 requested
   → Going to tool_node
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END
💬 The client has been identified as high risk due to their low balance of €750.

❓ What was the risk level you just assessed?
----------------------------------------
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END
💬 The risk level I assessed was HIGH RISK due to the client's low balance.

❓ Now check AML for a €12,000 transaction from Country_A for that same client
----------------------------------------
🤖 Agent thinking...
   Tool calls: 1 requested
   → Going to tool_node
🤖 Agent thinking...
   Tool calls: 0 requested
   → Going to END
💬 The transaction of €12,000 from Country_A has raised AML flags due to the high transaction amount and the country's high-risk status, requiring m